# Notebook 04 — SQL Analysis

**Goal:** Demonstrate SQL proficiency by loading the cleaned dataset into SQLite
and executing 8 analytical queries ranging from basic aggregations to CTEs and window functions.

**Queries:**
1. Overall churn rate
2. Churn by contract type
3. Average charges — churned vs retained
4. Top 5 segments by churn rate
5. Tenure cohort analysis (CASE WHEN)
6. Window function — rank by charges within contract
7. CTE — high-risk revenue impact
8. Subquery — compare each customer to contract-type average

In [ ]:
import sys
import warnings
import logging
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.sql_queries import load_into_sqlite, run_all_queries
import src.sql_queries as sq

sns.set_theme(style='whitegrid', palette='muted')
print('Ready.')

## Setup: Load Data into SQLite

In [ ]:
# Load raw cleaned data (with original string columns for SQL readability)
raw_clean = pd.read_csv(ROOT / 'data' / 'processed' / 'telco_churn_cleaned.csv')

# SQLite window functions require a version check
print('SQLite version:', sqlite3.sqlite_version)

conn, df_sql = load_into_sqlite(raw_clean)
print(f'Loaded {len(df_sql):,} rows into in-memory SQLite')

# Verify table
pd.read_sql_query('SELECT COUNT(*) AS n FROM customers', conn)

---
## Query 1 — Overall Churn Rate

In [ ]:
q1 = sq.q1_overall_churn_rate(conn)
print('Q1: Overall churn statistics')
q1

---
## Query 2 — Churn Rate by Contract Type

In [ ]:
q2 = sq.q2_churn_by_contract(conn)
print('Q2: Churn by contract type')
q2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#FF9800', '#2196F3', '#4CAF50']
bars = ax.bar(q2['Contract'], q2['churn_rate_pct'], color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, q2['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('SQL Q2: Churn Rate by Contract Type', fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'sql_q2_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Query 3 — Average Charges: Churned vs Retained

In [ ]:
q3 = sq.q3_avg_charges_churn_vs_retained(conn)
print('Q3: Average charges by churn status')
q3

---
## Query 4 — Top 5 Customer Segments by Churn Rate

In [ ]:
q4 = sq.q4_top_segments_by_churn_rate(conn)
print('Q4: Top 5 segments (contract × internet service) by churn rate')
q4

---
## Query 5 — Tenure Cohort Analysis (CASE WHEN)

In [ ]:
q5 = sq.q5_tenure_cohort_analysis(conn)
print('Q5: Churn rate by tenure bucket')
q5

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('viridis', len(q5))
bars = ax.bar(q5['tenure_bucket'], q5['churn_rate_pct'], color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, q5['churn_rate_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}%',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('SQL Q5: Tenure Cohort Churn Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'reports' / 'figures' / 'sql_q5_tenure_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Query 6 — Window Function: Rank Customers by Charges Within Contract Type

In [ ]:
q6 = sq.q6_rank_customers_by_charges(conn)
print('Q6: Top customers by monthly charge within each contract type (top 20)')
q6

---
## Query 7 — CTE: High-Risk Customer Revenue Impact

In [ ]:
q7 = sq.q7_high_risk_cte_revenue(conn)
print('Q7: High-risk segment (month-to-month, no tech support, tenure < 12 months)')
q7

In [ ]:
monthly_at_risk = q7['monthly_revenue_at_risk'].values[0]
annual_at_risk  = monthly_at_risk * 12
print(f'Monthly revenue at risk from high-risk segment: ${monthly_at_risk:,.0f}')
print(f'Annual revenue at risk:                         ${annual_at_risk:,.0f}')

---
## Query 8 — Subquery: Each Customer vs Contract-Type Average

In [ ]:
q8 = sq.q8_compare_to_contract_avg(conn)
print('Q8: Customers with largest deviation from their contract-type average charge')
q8

---
## Summary

In [ ]:
print('=== SQL ANALYSIS SUMMARY ===')
print(f"Total customers: {q1['total_customers'].values[0]:,}")
print(f"Total churned:   {q1['total_churned'].values[0]:,}")
print(f"Overall churn rate: {q1['churn_rate_pct'].values[0]}%")
print()
print('Contract type breakdown:')
for _, r in q2.iterrows():
    print(f"  {r['Contract']:20s}: {r['churn_rate_pct']:.1f}% churn ({r['churned']}/{r['total_customers']})")
print()
print(f"Monthly revenue at risk (high-risk segment): ${monthly_at_risk:,.0f}")
conn.close()
print('\nAll 8 SQL queries executed successfully.')